# DeBERTa-v3-small preference classifier

This is the first transformer experiment for the LLM preference task. It predicts probabilities for A win, B win, and tie. Start in `smoke` mode to verify the GPU, files, and training loop on a small stratified sample. If that succeeds, switch to `validation` for a full 70% train / 10% calibration / untouched 20% validation run. After reviewing that result, switch to `submission` to train on 90%, calibrate on 10%, and write `submission.csv`.

## Kaggle inputs required

1. Attach the competition data (`llm-classification-finetuning`).
2. Attach the public `microsoft/deberta-v3-small` model as a Kaggle Model input. The code searches `/kaggle/input` for its config and weights, so the folder name does not need to match a hard-coded path.
3. Enable a GPU accelerator. The notebook intentionally stops if no GPU is available.

The tokenizer decodes the JSON arrays into conversation turns. It encodes prompt + response A as the first segment and response B as the second, then truncates to the model's 512-token limit. This is our first simple truncation policy; we’ll compare its validation score before tuning it.

In [ ]:
# Imports, settings, and competition data
import json, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score
from scipy.optimize import minimize_scalar
from scipy.special import softmax

SEED = 42
MODE = "smoke"  # Run smoke first; then use "validation" and finally "submission".
SMOKE_FIT_ROWS = 2048
SMOKE_CAL_ROWS = 512
SMOKE_VALID_ROWS = 512
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUMULATION = 4
EPOCHS = 1
LEARNING_RATE = 2e-5
TARGET_COLUMNS = ["winner_model_a", "winner_model_b", "winner_tie"]

COMP_DIR = next((path.parent for path in Path("/kaggle/input").rglob("train.csv") if (path.parent / "test.csv").exists() and (path.parent / "sample_submission.csv").exists()), None)
assert COMP_DIR is not None, "Could not find the competition train.csv and test.csv files under /kaggle/input."
print("Competition folder:", COMP_DIR)
OUTPUT_DIR = Path("/kaggle/working")

# Kaggle mounts model inputs under varying directory names. Find the HF model folder.
def find_deberta_model_dir(root=Path("/kaggle/input")):
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        has_weights = any((config_path.parent / name).exists() for name in (
            "model.safetensors", "pytorch_model.bin", "model.safetensors.index.json", "pytorch_model.bin.index.json"
        ))
        if config.get("model_type") in {"deberta-v2", "deberta"} and has_weights:
            return config_path.parent
    raise FileNotFoundError("Could not find DeBERTa model config and weights under /kaggle/input. Attach the public microsoft/deberta-v3-small model input.")

MODEL_DIR = find_deberta_model_dir()
print("Model folder:", MODEL_DIR)

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator before running this notebook."
assert (COMP_DIR / "train.csv").exists(), f"Competition data not found at {COMP_DIR}"
assert (MODEL_DIR / "config.json").exists(), f"Model files not found at {MODEL_DIR}; attach the public model input and update MODEL_DIR."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Mode:", MODE)

def normalize_conversation(value):
    """Decode the CSV's JSON array of turn strings; keep turn boundaries."""
    if pd.isna(value):
        return ""
    try:
        parsed = json.loads(str(value))
    except (json.JSONDecodeError, TypeError):
        return str(value)
    if parsed is None:
        return ""
    if isinstance(parsed, list):
        return " [TURN] ".join("" if turn is None else str(turn).strip() for turn in parsed)
    return str(parsed)

train_df = pd.read_csv(COMP_DIR / "train.csv", usecols=["id", "prompt", "response_a", "response_b", *TARGET_COLUMNS])
labels = train_df[TARGET_COLUMNS].to_numpy().argmax(axis=1).astype(np.int64)
assert np.all(train_df[TARGET_COLUMNS].sum(axis=1).to_numpy() == 1), "Every row should have exactly one target label."
print(f"Training rows: {len(train_df):,}; label counts A/B/tie: {np.bincount(labels, minlength=3).tolist()}")

indices = np.arange(len(train_df))
if MODE == "smoke":
    smoke_size = SMOKE_FIT_ROWS + SMOKE_CAL_ROWS + SMOKE_VALID_ROWS
    selected_idx, _ = train_test_split(indices, train_size=smoke_size, random_state=SEED, stratify=labels)
    fit_cal_idx, valid_idx = train_test_split(selected_idx, test_size=SMOKE_VALID_ROWS, random_state=SEED + 1, stratify=labels[selected_idx])
    train_idx, cal_idx = train_test_split(fit_cal_idx, test_size=SMOKE_CAL_ROWS, random_state=SEED + 2, stratify=labels[fit_cal_idx])
elif MODE == "validation":
    fit_cal_idx, valid_idx = train_test_split(indices, test_size=0.20, random_state=SEED, stratify=labels)
    train_idx, cal_idx = train_test_split(fit_cal_idx, test_size=0.125, random_state=SEED + 1, stratify=labels[fit_cal_idx])
elif MODE == "submission":
    train_idx, cal_idx = train_test_split(indices, test_size=0.10, random_state=SEED + 1, stratify=labels)
    valid_idx = None
else:
    raise ValueError("MODE must be 'smoke', 'validation', or 'submission'.")

print(f"Fit rows: {len(train_idx):,}; calibration rows: {len(cal_idx):,}; validation rows: {0 if valid_idx is None else len(valid_idx):,}")

In [ ]:
# Tokenizer and memory-safe batched data loading

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False, local_files_only=True)
assert tokenizer.model_max_length >= MAX_LENGTH or tokenizer.model_max_length > 100000, "MAX_LENGTH exceeds tokenizer limit."

class PreferenceRows(Dataset):
    """Keep raw CSV text in memory; tokenize only the current batch."""
    def __init__(self, frame, row_labels=None):
        self.prompts = frame["prompt"].tolist()
        self.answers_a = frame["response_a"].tolist()
        self.answers_b = frame["response_b"].tolist()
        self.row_labels = None if row_labels is None else np.asarray(row_labels, dtype=np.int64)

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        first = f"Prompt: {normalize_conversation(self.prompts[index])} [ANSWER A] {normalize_conversation(self.answers_a[index])}"
        second = f"Answer B: {normalize_conversation(self.answers_b[index])}"
        label = None if self.row_labels is None else int(self.row_labels[index])
        return first, second, label

def preference_collate(batch):
    first, second, row_labels = zip(*batch)
    encoded = tokenizer(
        list(first),
        list(second),
        truncation="longest_first",
        max_length=MAX_LENGTH,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )
    if row_labels[0] is not None:
        encoded["labels"] = torch.tensor(row_labels, dtype=torch.long)
    return encoded

def make_loader(row_indices, batch_size, shuffle=False, include_labels=True):
    part = train_df.iloc[row_indices].reset_index(drop=True)
    y = labels[row_indices] if include_labels else None
    dataset = PreferenceRows(part, y)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=preference_collate,
        num_workers=0,
        pin_memory=True,
    )

train_loader = make_loader(train_idx, TRAIN_BATCH_SIZE, shuffle=True)
cal_loader = make_loader(cal_idx, EVAL_BATCH_SIZE, shuffle=False)
valid_loader = make_loader(valid_idx, EVAL_BATCH_SIZE, shuffle=False) if valid_idx is not None else None
print(f"Memory-safe loaders: {len(train_loader):,} fitting batches; {len(cal_loader):,} calibration batches")

In [ ]:
# Fine-tune the pretrained encoder for one learning epoch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=3,
    id2label={0: TARGET_COLUMNS[0], 1: TARGET_COLUMNS[1], 2: TARGET_COLUMNS[2]},
    label2id={name: i for i, name in enumerate(TARGET_COLUMNS)},
    local_files_only=True,
    ignore_mismatched_sizes=True,
).to(device).float()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=True)

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    for step, batch in enumerate(train_loader):
        y = batch.pop("labels").to(device)
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.float16):
            loss = model(**batch, labels=y).loss / GRAD_ACCUMULATION
        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUMULATION
        if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        if (step + 1) % 200 == 0:
            print(f"epoch {epoch + 1}/{EPOCHS}, batch {step + 1}/{len(train_loader)}, mean loss={running_loss / (step + 1):.4f}")
    print(f"Epoch {epoch + 1} mean training loss: {running_loss / len(train_loader):.4f}")

def predict_logits(loader):
    model.eval()
    output = []
    with torch.no_grad():
        for batch in loader:
            batch.pop("labels", None)
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.float16):
                logits = model(**batch).logits
            output.append(logits.float().cpu().numpy())
    return np.concatenate(output, axis=0)

cal_logits = predict_logits(cal_loader)
y_cal = labels[cal_idx]

def probs_at_temperature(logits, temperature):
    return softmax(logits / temperature, axis=1)

result = minimize_scalar(
    lambda log_t: log_loss(y_cal, probs_at_temperature(cal_logits, np.exp(log_t)), labels=[0, 1, 2]),
    bounds=(np.log(0.25), np.log(8.0)),
    method="bounded",
)
temperature = float(np.exp(result.x))
print(f"Calibration temperature: {temperature:.3f}")

In [ ]:
# Evaluate, or make the offline competition submission

if MODE in {"smoke", "validation"}:
    valid_logits = predict_logits(valid_loader)
    raw_probs = softmax(valid_logits, axis=1)
    calibrated_probs = probs_at_temperature(valid_logits, temperature)
    y_valid = labels[valid_idx]
    prior = np.bincount(labels[train_idx], minlength=3) / len(train_idx)
    print(f"Uniform reference log loss: {log_loss(y_valid, np.full_like(calibrated_probs, 1/3), labels=[0,1,2]):.5f}")
    print(f"Training-prior reference log loss: {log_loss(y_valid, np.tile(prior, (len(y_valid), 1)), labels=[0,1,2]):.5f}")
    print(f"Raw validation log loss: {log_loss(y_valid, raw_probs, labels=[0,1,2]):.5f}")
    print(f"Calibrated validation log loss: {log_loss(y_valid, calibrated_probs, labels=[0,1,2]):.5f}")
    print(f"Validation accuracy: {accuracy_score(y_valid, calibrated_probs.argmax(axis=1)):.4f}")
    if MODE == "smoke":
        print("Smoke run complete. If this cell ran successfully, change MODE to 'validation' for the full holdout run.")
else:
    test_df = pd.read_csv(COMP_DIR / "test.csv", usecols=["id", "prompt", "response_a", "response_b"])
    test_dataset = PreferenceRows(test_df)
    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=preference_collate,
        num_workers=0,
        pin_memory=True,
    )
    test_logits = predict_logits(test_loader)
    test_probs = softmax(test_logits, axis=1)
    submission = pd.DataFrame(test_probs, columns=TARGET_COLUMNS)
    submission.insert(0, "id", test_df["id"])
    assert np.allclose(submission[TARGET_COLUMNS].sum(axis=1), 1.0, atol=1e-4)
    submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("Wrote:", OUTPUT_DIR / "submission.csv", "rows:", len(submission))
    display(submission.head())